In [3]:
# import
import torch
import random
import torch.nn as nn
import torch.nn.functional as F

import warnings
warnings.filterwarnings("ignore")

seed = 333

# Hyperparameter

In [4]:
hp = {
    "t":8,
    "c":32,
}

# Data

## Import Data

In [5]:
# import data
with open("../data/input.txt","r",newline="\n") as file :
    data_file = file.read()

data_file[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

## Data Class

In [6]:
class Data:
    def __init__(
        self,
        _data: str,
        seed: int = seed,
        train_batch_size: int = None,
        train_block_size: int = None,
        test_batch_size: int = None,
        test_block_size: int = None,
    ) -> None:
        # initialisation
        self.seed = seed
        self.data = _data  # raw data without process
        self.len_data = len(_data)  # how many character in raw data
        self.char_list = sorted(list(set(_data)))  # list of possible character
        self.len_char = len(self.char_list)  # len of possible character

        # encoder and decoder list
        self.enc_dict = {key: value for value, key in enumerate(self.char_list)}  # convert to number
        self.dec_dict = {key: value for key, value in enumerate(self.char_list)}  # convert to string

        # encoded data
        self.data_enc = self.encoder(self.data)  # raw data without process

        # split to train and test set with ration 9:1
        split_at_index = self.len_data // 9
        self.train = self.data_enc[:split_at_index]  # full train set
        self.test = self.data_enc[split_at_index:]  # full test set

        # create x,y for train and test set
        if not None in [train_batch_size, test_batch_size, train_block_size, test_block_size]:
            self.x_train, self.y_train = self.sample_batch(
                batch_size=train_batch_size, block_size=train_block_size, input=self.train
            )
            self.x_test, self.y_test = self.sample_batch(
                batch_size=test_batch_size, block_size=test_block_size, input=self.test
            )

    def encoder(self, input: str) -> torch.Tensor:
        """convert string to encoded index

        Args:
            input (str): string to be encoded

        Returns:
            torch.Tensor: (copy) tensor of encoded index
        """
        return torch.tensor([self.enc_dict[char] for char in input])

    def decoder(self, input: torch.Tensor) -> str:
        """covnert encoded index in torch.tensor into

        Args:
            input (torch.Tensor): tensor of encoded index

        Returns:
            str: (copy) string from encoded index in tensor(input)
        """
        return "".join([self.dec_dict[i.item()] for i in input])

    def sample_batch(self, batch_size: int, block_size: int, input: torch.Tensor):
        """generate sample batch

        Args:
            batch_size (int): how many batch generated
            block_size (int): maximum length per batch
            input (torch.Tensor): tensor of encoded data

        Returns:
            x,y: data_x,data_y
        """
        random.seed(self.seed)
        output_x = torch.empty((batch_size, block_size), dtype=torch.long)
        output_y = torch.empty((batch_size, block_size), dtype=torch.long) if block_size !=1 else torch.empty((batch_size), dtype=torch.long)
        len_input = len(input)

        for batch_index in range(batch_size):
            offset = random.randint(0, len_input - block_size - 1)
            output_x[batch_index] = input[offset : offset + block_size]
            output_y[batch_index] = input[offset + 1 : offset + block_size + 1]
        return output_x, output_y

In [7]:
d = Data(
    _data=data_file,
    seed=seed,
    train_batch_size=50000,
    test_batch_size=50000,
    train_block_size=hp["t"],
    test_block_size=hp["t"],
)

# Model

In [8]:
class Head(nn.Module) :
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        torch.manual_seed(seed)
        
        # layers
        self.pos_emb = nn.Embedding(hp["t"],hp["c"])
        self.lin1 = nn.Linear(hp["c"],hp["c"])
        self.value = nn.Embedding(hp["t"],hp["t"])
        
    def forward(self,input:torch.Tensor) -> torch.Tensor:
        """forward prop

        Args:
            input (torch.Tensor): feed in embedded input data,shape(b,t,c)

        Returns:
            torch.Tensor: output tensor with shape # (b,t,t)
        """
        # get output
        pos_emb_out = self.pos_emb(torch.arange(hp["t"])) # (t,c)
        lin1_out = self.lin1(input) # (b,t,c)
        value_out = self.value(torch.arange(hp["t"])) # (t,t)
        
        # process output
        ep = input + pos_emb_out # (b,t,c)
        value_out_mask = value_out.masked_fill(torch.tril(torch.ones(hp["t"],hp["t"]))==0,float("-inf"))
        value_out_softmax = F.softmax(value_out_mask,dim=1) # (t,t)
        square_matrix = ep @ lin1_out.transpose(1,2) # (b,t,t)
        out = value_out_softmax @ square_matrix # (b,t,t)
        
        return out

In [9]:
class Model(nn.Module):
    def __init__(self):
        # initialisation
        super().__init__()
        torch.manual_seed(seed)

        # embedding
        self.embedding = nn.Embedding(d.len_char, d.len_char)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        """forward prop

        Args:
            input (torch.Tensor): input tensor with shape (batch_size,block_size=1)

        Returns:
            torch.Tensor: logit tensor with shape (batch_size,len_char=65)
        """
        logits: torch.Tensor = self.embedding(input)

        # squeeze into 2-d tensor if dim_1 == 1
        if (logits.ndim == 3) and (logits.shape[1] == 1):
            logits = torch.squeeze(logits, dim=1)

        return logits

    def loss(self, logits: torch.Tensor, expected: torch.Tensor):
        """calculate loss

        Args:
            logits (torch.Tensor): logit tensor with size (batch_size,len_char)
            expected (torch.Tensor): expected class tensor with shape (batch size,categorical number within len_char)

        Returns:
            loss_value in tensor
        """
        return F.cross_entropy(logits, expected)

    @torch.no_grad()
    def loss_test(self):
        """calculate loss of test set

        Returns:
            loss value of test set
        """
        return self.loss(self(d.x_test), d.y_test).item()

    def fit(
        self,
        x: torch.Tensor,
        y: torch.Tensor,
        lr: float = 1e-3,
        batch_size: int = 100,
        iteration: int = 10000,
        print_iteration: int = 1000,
    ) -> None:
        # optimizer
        optimiser = torch.optim.Adam(self.parameters(), lr=lr)
        for i in range(iteration):

            # batch x,y
            batch_index = torch.randint(low=0, high=y.shape[0], size=(batch_size,))
            batch_x = x[batch_index]
            batch_y = y[batch_index]

            loss = self.loss(logits=self(batch_x), expected=batch_y)
            optimiser.zero_grad(set_to_none=True)
            loss.backward()
            optimiser.step()

            if ((i + 1) % print_iteration == 0) or ((i + 1) == iteration) or (i == 0):
                if i==0 :
                    print("                      Loss_Train : Loss test")
                with torch.no_grad():
                    print(f"{i+1}th Iteration = {self.loss(self.forward(x), y):.4f} : {self.loss_test():.4f}")
    
    @torch.no_grad()
    def sampling(self, number_of_word: int):
        out = torch.randint(low=0,high=d.len_char,size=(1,))

        for _ in range(number_of_word):
            # get last char
            last_char_enc = out[-1]

            # forward
            logits = self(last_char_enc)
            softmax = F.softmax(logits)

            # get next char
            next_char_enc = torch.multinomial(softmax, 1)

            # append
            out = torch.cat((out,next_char_enc))
        print(d.decoder(out))

In [ ]:
m = Model()
m.fit(
    x=d.x_train,
    y=d.y_train,
    lr=1e-3,
    batch_size=100,
    iteration=10000,
    print_iteration=1000,
)

In [ ]:
m.sampling(5000)